# NB18 — CALIBRATION, REGRESSION-TO-THE-MEAN, STORAGE-PHASE TABLES AND SVR (C, γ) SURFACE | NIR-HUEVOS 2026

**CPU-only, no model training, ~5–10 min. It only reads frozen out-of-fold predictions and search logs; nothing is refitted.**

## What it resolves
1. **Attenuation vs calibration (reviewer 1, §3.3/§4.4).** The predicted-on-observed slope (Supplementary Table S3) is below 1 for *any* imperfect
   regression model, because ŷ regresses toward the mean of the target. The operationally relevant question is the reverse one: *given a prediction,
   what is the expected true storage time?* NB18 reports both slopes (identity: slope(ŷ~y) × slope(y~ŷ) = line R²), egg-cluster bootstrap CIs, and
   decomposes the late-phase bias into the part implied by the linear slope and the remainder (curvature/saturation).
2. **Calibration curves** (binned prediction vs mean observed age) with egg-cluster CIs.
3. **Day-specific bias and MAE with 95% egg-cluster bands** (redraws manuscript Figure 5 with uncertainty) and a corrected **Figure 3**
   (the y-axis label of the current file is clipped) showing both regression lines.
4. **Storage-phase classification for every model, including CNN1D** (extends Table 7 / Supplementary Table S2). The implementation is validated
   against the frozen `NB09B_storage_phase_classification_summary.csv` for the six models that already have it.
5. **SVR (C, γ) surface** from the frozen inner-search logs (NB03 primary grid and NB09B wider grid) to show whether the boundary selections sit on a plateau.

## Inputs (found by name under the project root)
`NB12_unified_oof_predictions.csv` (required) · `NB16_oof_predictions_seedmean.csv` (optional; adds CNN1D_FLAT / CNN1D_GAP_POS) ·
`NB03_inner_search_summary.csv` · `NB09B_SVR_sensitivity_inner_search.csv` · `NB09B_storage_phase_classification_summary.csv` (validation).

## Output
`05_RESULTS/REVISION_REVIEWERS_2026_09/NB18_CALIBRATION_PHASES_AND_SVR_SURFACE/` and a ZIP in `05_RESULTS/ZIP_PACKAGES/` (downloaded by the last cell).

In [1]:
import os, sys, json, shutil, platform, subprocess, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except Exception:
    IN_COLAB = False

QUICK_TEST = os.environ.get('NIR_QUICK_TEST', '0') == '1'
PROJECT_ROOT = Path(os.environ.get('NIR_PROJECT_ROOT', '/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026'))
RES_ROOT = PROJECT_ROOT / '05_RESULTS'
RESULT_DIR = RES_ROOT / 'REVISION_REVIEWERS_2026_09' / 'NB18_CALIBRATION_PHASES_AND_SVR_SURFACE'
ZIP_DIR = RES_ROOT / 'ZIP_PACKAGES'
for p in [RESULT_DIR, ZIP_DIR]: p.mkdir(parents=True, exist_ok=True)
RUN_REVISION = 'NB18_v1_calibration_phase_tables_svr_surface'
BOOT_REPS, BOOT_SEED = (300 if QUICK_TEST else 10000), 20260915
PHASE_EDGES = (7.5, 14.5)                      # Early 0-7 | Middle 8-14 | Late 15-21 (manuscript definition)
PHASE_NAMES = ['Early (0-7)', 'Middle (8-14)', 'Late (15-21)']
PHASE_OF_DAY = lambda d: np.where(d <= 7, 0, np.where(d <= 14, 1, 2))
DAYS = np.arange(22)
OKABE = {'SVR': '#0072B2', 'PLSR': '#E69F00', 'ANN': '#009E73', 'CNN1D': '#CC79A7', 'CNN1D_FLAT': '#D55E00',
         'CNN1D_GAP_POS': '#56B4E9', 'BiLSTM': '#000000', 'LSTM': '#666666', 'SimpleRNN': '#999999'}
MARK = {'SVR': 'o', 'PLSR': 's', 'ANN': 'D', 'CNN1D': '^', 'CNN1D_FLAT': 'v', 'CNN1D_GAP_POS': 'P'}
LSTY = {'SVR': '-', 'PLSR': '--', 'ANN': '-.', 'CNN1D': ':', 'CNN1D_FLAT': (0, (5, 1)), 'CNN1D_GAP_POS': (0, (1, 1))}
plt.rcParams.update({'font.family': 'serif', 'font.size': 9, 'axes.linewidth': 0.8, 'savefig.dpi': 600})

def find_one(filename, root=PROJECT_ROOT, required=True):
    hits = sorted(root.rglob(filename))
    if not hits:
        assert not required, f'Not found anywhere under {root}: {filename}'
        return None
    if len(hits) > 1: print(f'  note: {len(hits)} copies of {filename}; using {hits[0].relative_to(root)}')
    return hits[0]

def save_fig(fig, stem):
    png = RESULT_DIR / f'{stem}.png'; tif = RESULT_DIR / f'{stem}.tif'
    fig.savefig(png, dpi=600, facecolor='white', bbox_inches='tight')
    try:
        from PIL import Image
        Image.open(png).convert('RGB').save(tif, compression='tiff_lzw', dpi=(600, 600))
    except Exception as e:
        print('  TIFF not written:', e)
    plt.close(fig)
    print('  figure written:', png.name)
print('Setup OK | bootstrap reps:', BOOT_REPS, '| project root exists:', PROJECT_ROOT.exists())

Mounted at /content/drive
Setup OK | bootstrap reps: 10000 | project root exists: True


In [2]:
# ------------------------------ load frozen OOF -------------------------------
uni = pd.read_csv(find_one('NB12_unified_oof_predictions.csv'))
oof = uni[['sample', 'storage_days', 'model', 'y_pred']].copy()
f16 = find_one('NB16_oof_predictions_seedmean.csv', required=False)
if f16 is not None:
    o16 = pd.read_csv(f16)[['sample', 'storage_days', 'model', 'y_pred']]
    oof = pd.concat([oof, o16], ignore_index=True)
    print('NB16 predictions added:', sorted(o16['model'].unique()))
MODELS = [m for m in ['SVR', 'PLSR', 'ANN', 'CNN1D', 'CNN1D_FLAT', 'CNN1D_GAP_POS', 'BiLSTM', 'LSTM', 'SimpleRNN'] if m in set(oof['model'])]
KEY_MODELS = [m for m in ['SVR', 'PLSR', 'ANN'] if m in MODELS]
CNN_MODELS = [m for m in ['CNN1D', 'CNN1D_FLAT', 'CNN1D_GAP_POS'] if m in MODELS]
print('Models:', MODELS)

def egg_arrays(oof_long, models):
    eggs = sorted(oof_long['sample'].unique())
    Y = None; P = []
    for m in models:
        d = oof_long[oof_long['model'] == m].sort_values(['sample', 'storage_days'])
        assert len(d) == 660 and d.groupby('sample').size().eq(22).all(), f'{m}: expected 30 eggs x 22 days'
        P.append(d['y_pred'].to_numpy(float).reshape(len(eggs), 22))
        if Y is None: Y = d['storage_days'].to_numpy(float).reshape(len(eggs), 22)
    return eggs, Y, np.stack(P)

EGGS, Y, P = egg_arrays(oof, MODELS)
E = len(EGGS); mi = {m: i for i, m in enumerate(MODELS)}
rng = np.random.default_rng(BOOT_SEED)
IDX = rng.integers(0, E, size=(BOOT_REPS, E))          # one shared set of whole-egg resamples for every analysis below
print('Arrays:', Y.shape, P.shape, '| bootstrap index matrix:', IDX.shape)

  note: 2 copies of NB12_unified_oof_predictions.csv; using 05_RESULTS/REVISION_REVIEWERS_2026_09/NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE/NB12_unified_oof_predictions.csv
NB16 predictions added: ['CNN1D_FLAT', 'CNN1D_GAP_POS']
Models: ['SVR', 'PLSR', 'ANN', 'CNN1D', 'CNN1D_FLAT', 'CNN1D_GAP_POS', 'BiLSTM', 'LSTM', 'SimpleRNN']
Arrays: (30, 22) (9, 30, 22) | bootstrap index matrix: (10000, 30)


## 1. Attenuation versus calibration
For pooled out-of-fold rows, let *y* be observed and *ŷ* predicted storage time.
* slope(ŷ~y) — the "attenuation" slope of Supplementary Table S3 (equal to ≈ R² for a calibrated predictor).
* slope(y~ŷ) — the **calibration slope**; 1 means that a prediction of *x* days corresponds on average to *x* observed days.
* Identity: slope(ŷ~y) × slope(y~ŷ) = r² (line R²). It is asserted below as an integrity check.

In [3]:
def line_stats(Yb, Pb):
    """Vectorised OLS statistics on pooled rows. Yb, Pb: (..., E, 22)."""
    ax = (-2, -1)
    my = Yb.mean(axis=ax); mp = Pb.mean(axis=ax)
    dy = Yb - my[..., None, None]; dp = Pb - mp[..., None, None]
    cov = (dy * dp).mean(axis=ax); vy = (dy ** 2).mean(axis=ax); vp = (dp ** 2).mean(axis=ax)
    a1 = cov / vy; a0 = mp - a1 * my            # yhat = a0 + a1*y
    b1 = cov / vp; b0 = my - b1 * mp            # y    = b0 + b1*yhat
    r2 = cov ** 2 / (vy * vp)
    err = Pb - Yb
    R2 = 1 - (err ** 2).sum(axis=ax) / ((Yb - my[..., None, None]) ** 2).sum(axis=ax)
    return dict(slope_pred_on_obs=a1, intercept_pred_on_obs=a0, slope_obs_on_pred=b1, intercept_obs_on_pred=b0,
                line_R2=r2, pooled_R2=R2, bias=err.mean(axis=ax))

def ci(a, q=(2.5, 97.5)): return np.percentile(a, q)

rows = []; boot_store = {}
for m in MODELS:
    pt = line_stats(Y, P[mi[m]])
    assert abs(pt['slope_pred_on_obs'] * pt['slope_obs_on_pred'] - pt['line_R2']) < 1e-9
    bs = {k: [] for k in pt}
    for s in range(0, BOOT_REPS, 500):
        idx = IDX[s:s + 500]
        r = line_stats(Y[idx], P[mi[m]][idx])
        for k in bs: bs[k].append(r[k])
    bs = {k: np.concatenate(v) for k, v in bs.items()}; boot_store[m] = bs
    row = {'model': m}
    for k in ['slope_pred_on_obs', 'slope_obs_on_pred', 'intercept_pred_on_obs', 'intercept_obs_on_pred', 'line_R2', 'pooled_R2', 'bias']:
        lo, hi = ci(bs[k]); row.update({k: float(pt[k]), f'{k}_CI_low': float(lo), f'{k}_CI_high': float(hi)})
    row['slope_obs_on_pred_minus_1'] = row['slope_obs_on_pred'] - 1
    rows.append(row)
calib = pd.DataFrame(rows); calib.to_csv(RESULT_DIR / 'Table_NB18_calibration_slopes.csv', index=False)
display(calib[['model', 'slope_pred_on_obs', 'slope_pred_on_obs_CI_low', 'slope_pred_on_obs_CI_high',
               'slope_obs_on_pred', 'slope_obs_on_pred_CI_low', 'slope_obs_on_pred_CI_high', 'line_R2', 'pooled_R2', 'bias']].round(3))
print('Identity slope(yhat~y) x slope(y~yhat) = line R2 verified for every model.')

# cross-check against the frozen Supplementary Table S3 source (NB09B), when present
f_s3 = find_one('NB09B_predicted_on_observed_attenuation_slopes.csv', required=False)
if f_s3 is not None:
    s3 = pd.read_csv(f_s3).set_index('model')
    for m in KEY_MODELS:
        d = abs(calib.set_index('model').loc[m, 'slope_pred_on_obs'] - s3.loc[m, 'predicted_on_observed_slope'])
        print(f'  S3 check {m}: |slope difference| = {d:.2e}')
        if not QUICK_TEST: assert d < 1e-6, f'{m}: slope differs from frozen NB09B value'

,model,slope_pred_on_obs,slope_pred_on_obs_CI_low,slope_pred_on_obs_CI_high,slope_obs_on_pred,slope_obs_on_pred_CI_low,slope_obs_on_pred_CI_high,line_R2,pooled_R2,bias
0,SVR,0.855,0.824,0.885,0.958,0.924,0.995,0.819,0.817,0.148
1,PLSR,0.845,0.820,0.872,0.945,0.899,0.994,0.799,0.796,0.006
2,ANN,0.823,0.787,0.860,0.952,0.895,1.009,0.783,0.780,-0.159
3,CNN1D,0.347,0.273,0.419,0.724,0.554,0.963,0.251,0.214,0.022
4,CNN1D_FLAT,0.771,0.734,0.810,0.973,0.882,1.061,0.750,0.746,-0.359
5,CNN1D_GAP_POS,0.264,0.216,0.312,0.760,0.568,1.059,0.200,0.168,0.695
6,BiLSTM,0.287,0.243,0.330,0.833,0.636,1.145,0.239,0.225,-0.428
7,LSTM,0.215,0.180,0.252,1.075,0.923,1.247,0.231,0.229,-0.225
8,SimpleRNN,0.175,0.144,0.207,1.075,0.830,1.421,0.188,0.187,0.174


Identity slope(yhat~y) x slope(y~yhat) = line R2 verified for every model.
  S3 check SVR: |slope difference| = 0.00e+00
  S3 check PLSR: |slope difference| = 2.22e-16
  S3 check ANN: |slope difference| = 1.11e-16


In [4]:
# ------------- phase-wise bias: observed vs the part implied by the linear slope -------------
phase_days = {'Early (0-7)': np.arange(0, 8), 'Middle (8-14)': np.arange(8, 15), 'Late (15-21)': np.arange(15, 22)}
def phase_bias(Yb, Pb, dset):
    sel = np.isin(DAYS, dset)
    yb = Yb[..., sel]; pb = Pb[..., sel]
    return (pb - yb).mean(axis=(-2, -1)), yb.mean(axis=(-2, -1))

rows = []
for m in MODELS:
    ls = boot_store[m]
    for ph, dset in phase_days.items():
        obs_pt, ybar = phase_bias(Y, P[mi[m]], dset)
        pt = line_stats(Y, P[mi[m]])
        implied_pt = pt['intercept_pred_on_obs'] + pt['slope_pred_on_obs'] * ybar - ybar
        # bootstrap
        obs_b, yb_b = phase_bias(Y[IDX], P[mi[m]][IDX], dset)
        implied_b = ls['intercept_pred_on_obs'] + ls['slope_pred_on_obs'] * yb_b - yb_b
        resid_b = obs_b - implied_b
        rows.append({'model': m, 'phase': ph, 'mean_observed_day': float(ybar),
                     'observed_bias_days': float(obs_pt), 'observed_CI_low': float(ci(obs_b)[0]), 'observed_CI_high': float(ci(obs_b)[1]),
                     'bias_implied_by_linear_slope_days': float(implied_pt), 'implied_CI_low': float(ci(implied_b)[0]),
                     'implied_CI_high': float(ci(implied_b)[1]), 'residual_beyond_linear_days': float(obs_pt - implied_pt),
                     'residual_CI_low': float(ci(resid_b)[0]), 'residual_CI_high': float(ci(resid_b)[1]),
                     'share_explained_by_linear_slope': float(implied_pt / obs_pt) if abs(obs_pt) >= 0.25 else np.nan})
phase_bias_tab = pd.DataFrame(rows); phase_bias_tab.to_csv(RESULT_DIR / 'Table_NB18_phase_bias_decomposition.csv', index=False)
display(phase_bias_tab[phase_bias_tab['model'].isin(KEY_MODELS)].round(3))

,model,phase,mean_observed_day,observed_bias_days,observed_CI_low,observed_CI_high,bias_implied_by_linear_slope_days,implied_CI_low,implied_CI_high,residual_beyond_linear_days,residual_CI_low,residual_CI_high,share_explained_by_linear_slope
0,SVR,Early (0-7),3.5,0.905,0.438,1.356,1.166,0.711,1.620,-0.261,-0.391,-0.132,1.288
1,SVR,Middle (8-14),11.0,0.623,0.096,1.154,0.075,-0.388,0.540,0.548,0.310,0.792,0.121
2,SVR,Late (15-21),18.0,-1.192,-1.769,-0.595,-0.943,-1.505,-0.370,-0.249,-0.360,-0.141,0.791
3,PLSR,Early (0-7),3.5,0.836,0.334,1.331,1.088,0.591,1.596,-0.252,-0.402,-0.106,1.301
4,PLSR,Middle (8-14),11.0,0.548,-0.074,1.237,-0.071,-0.587,0.482,0.619,0.339,0.907,-0.130
5,PLSR,Late (15-21),18.0,-1.484,-2.071,-0.849,-1.153,-1.750,-0.507,-0.331,-0.468,-0.193,0.777
6,ANN,Early (0-7),3.5,0.782,0.414,1.161,1.083,0.728,1.461,-0.301,-0.452,-0.139,1.385
7,ANN,Middle (8-14),11.0,0.362,-0.346,1.101,-0.247,-0.764,0.300,0.609,0.281,0.929,-0.684
8,ANN,Late (15-21),18.0,-1.754,-2.423,-1.052,-1.489,-2.215,-0.728,-0.265,-0.431,-0.098,0.849


## 2. Calibration curves, day-specific bias/MAE with uncertainty, and corrected Figures 3 and 5

In [5]:
# ---- binned calibration curve (bins of the prediction; whole-egg bootstrap for the mean observed age) ----
N_BINS = 7
def calib_curve(m):
    p = P[mi[m]]; edges = np.quantile(p.ravel(), np.linspace(0, 1, N_BINS + 1)); edges[0] -= 1e-9; edges[-1] += 1e-9
    b = np.digitize(p, edges[1:-1])                                   # (E,22) bin id 0..N_BINS-1
    def stat(idx):
        yy = Y[idx]; pp = p[idx]; bb = b[idx]
        return np.array([[pp[bb == k].mean(), yy[bb == k].mean()] if (bb == k).any() else [np.nan, np.nan] for k in range(N_BINS)])
    pt = stat(np.arange(E))
    boots = np.stack([stat(IDX[i]) for i in range(min(BOOT_REPS, 2000))])
    lo, hi = np.nanpercentile(boots[:, :, 1], [2.5, 97.5], axis=0)
    return pd.DataFrame({'model': m, 'bin': np.arange(1, N_BINS + 1), 'mean_predicted': pt[:, 0], 'mean_observed': pt[:, 1],
                         'observed_CI_low': lo, 'observed_CI_high': hi, 'n_rows': [int((b == k).sum()) for k in range(N_BINS)]})
cc = pd.concat([calib_curve(m) for m in MODELS], ignore_index=True)
cc.to_csv(RESULT_DIR / 'Table_NB18_calibration_curve_bins.csv', index=False)

fig, axes = plt.subplots(1, len(KEY_MODELS), figsize=(6.5, 2.7), sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, KEY_MODELS):
    d = cc[cc['model'] == m]
    ax.plot([-1, 23], [-1, 23], color='#555555', lw=0.9, ls='--')
    ax.errorbar(d['mean_predicted'], d['mean_observed'], yerr=[d['mean_observed'] - d['observed_CI_low'], d['observed_CI_high'] - d['mean_observed']],
                fmt=MARK[m], color=OKABE[m], ms=4, capsize=2, lw=1)
    row = calib[calib['model'] == m].iloc[0]
    ax.text(0.97, 0.04, f"{m}\ncalibration slope = {row['slope_obs_on_pred']:.3f}", transform=ax.transAxes, ha='right', va='bottom', fontsize=7.5)
    ax.grid(alpha=0.2)
axes[0].set_xlim(-1, 23); axes[0].set_ylim(-1, 23)
fig.supxlabel('Mean predicted storage time within bin (days)', fontsize=9)
fig.supylabel('Mean observed storage time (days)', fontsize=9)
save_fig(fig, 'NB18_Figure_calibration_curves')

  figure written: NB18_Figure_calibration_curves.png


In [6]:
# ---- day-specific bias and MAE with 95% whole-egg bootstrap bands ----
rows = []
for m in MODELS:
    e = P[mi[m]] - Y                                   # (E,22)
    bias_pt = e.mean(axis=0); mae_pt = np.abs(e).mean(axis=0)
    bias_b = (P[mi[m]][IDX] - Y[IDX]).mean(axis=1)     # (B,22)
    mae_b = np.abs(P[mi[m]][IDX] - Y[IDX]).mean(axis=1)
    lo_b, hi_b = np.percentile(bias_b, [2.5, 97.5], axis=0); lo_m, hi_m = np.percentile(mae_b, [2.5, 97.5], axis=0)
    for d in DAYS:
        rows.append({'model': m, 'storage_day': int(d), 'bias_days': bias_pt[d], 'bias_CI_low': lo_b[d], 'bias_CI_high': hi_b[d],
                     'MAE_days': mae_pt[d], 'MAE_CI_low': lo_m[d], 'MAE_CI_high': hi_m[d]})
byday = pd.DataFrame(rows); byday.to_csv(RESULT_DIR / 'Table_NB18_metrics_by_storage_day_with_CI.csv', index=False)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.8), constrained_layout=True)
for m in KEY_MODELS:
    d = byday[byday['model'] == m]
    a1.plot(d['storage_day'], d['MAE_days'], color=OKABE[m], ls=LSTY[m], marker=MARK[m], ms=3, lw=1.1, label=m)
    a2.plot(d['storage_day'], d['bias_days'], color=OKABE[m], ls=LSTY[m], marker=MARK[m], ms=3, lw=1.1, label=m)
    if m == 'SVR':   # band shown only for the model highlighted in the text, to keep the panels legible
        a1.fill_between(d['storage_day'], d['MAE_CI_low'], d['MAE_CI_high'], color=OKABE[m], alpha=0.15, lw=0)
        a2.fill_between(d['storage_day'], d['bias_CI_low'], d['bias_CI_high'], color=OKABE[m], alpha=0.15, lw=0)
a2.axhline(0, color='#666666', lw=0.8)
a1.set_xlabel('Storage time (days)'); a2.set_xlabel('Storage time (days)')
a1.set_ylabel('MAE (days)'); a2.set_ylabel('Bias (days)'); a1.legend(frameon=False, fontsize=8); a1.grid(alpha=0.2); a2.grid(alpha=0.2)
save_fig(fig, 'NB18_Figure5_v2_MAE_and_bias_by_day_with_SVR_band')

# ---- corrected Figure 3: both regression lines; the y label is a figure-level label so it cannot be clipped ----
fig, axes = plt.subplots(1, len(KEY_MODELS), figsize=(6.6, 2.6), sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
xs = np.array([0, 21])
for ax, m in zip(axes, KEY_MODELS):
    r = calib[calib['model'] == m].iloc[0]
    jit = (np.random.default_rng(1).uniform(-0.15, 0.15, size=Y.shape))
    ax.scatter((Y + jit).ravel(), P[mi[m]].ravel(), s=5, alpha=0.30, color='#3182bd', edgecolors='none', rasterized=True)
    ax.plot(xs, xs, color='#555555', lw=0.9, ls='--', label='identity')
    ax.plot(xs, r['intercept_pred_on_obs'] + r['slope_pred_on_obs'] * xs, color='#D55E00', lw=1.4, label='predicted on observed')
    yy = np.array([-2, 24]); ax.plot(r['intercept_obs_on_pred'] + r['slope_obs_on_pred'] * yy, yy, color='#009E73', lw=1.4, ls='-', label='observed on predicted')
    ax.text(0.97, 0.05, f"{m}\nslope (pred~obs) = {r['slope_pred_on_obs']:.3f}\nslope (obs~pred) = {r['slope_obs_on_pred']:.3f}",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=7)
    ax.set_xlabel('Observed storage time (days)'); ax.grid(alpha=0.15)
axes[0].set_ylim(-10, 28); axes[0].set_xlim(-1, 22.5)
fig.supylabel('OOF predicted storage time (days)', fontsize=9)
axes[0].legend(frameon=False, fontsize=6.5, loc='upper left')
save_fig(fig, 'NB18_Figure3_v2_observed_vs_predicted_both_slopes')

  figure written: NB18_Figure5_v2_MAE_and_bias_by_day_with_SVR_band.png
  figure written: NB18_Figure3_v2_observed_vs_predicted_both_slopes.png


## 3. Storage-phase classification for every model (extends Table 7 and Supplementary Table S2)
Predicted phase = threshold of the **unclipped** predicted storage time at 7.5 and 14.5 days (Early 0–7, Middle 8–14, Late 15–21).
The six models already in NB09B are used as a validation of this implementation.

In [7]:
def phase_table(m):
    yt = PHASE_OF_DAY(Y.ravel()); yp = np.digitize(P[mi[m]].ravel(), PHASE_EDGES)
    cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
    n = cm.sum(); per = []
    for k, name in enumerate(PHASE_NAMES):
        tp = cm[k, k]; fn = cm[k].sum() - tp; fp = cm[:, k].sum() - tp; tn = n - tp - fn - fp
        sens = tp / (tp + fn) if tp + fn else np.nan; spec = tn / (tn + fp) if tn + fp else np.nan
        prec = tp / (tp + fp) if tp + fp else 0.0; f1 = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
        per.append({'model': m, 'storage_phase': name, 'sensitivity': sens, 'specificity': spec, 'precision': prec, 'F1': f1, 'n': int(tp + fn)})
    per = pd.DataFrame(per); sup = per['n'].to_numpy()
    summ = {'model': m, 'accuracy': float(np.trace(cm) / n), 'balanced_accuracy': float(per['sensitivity'].mean()),
            'macro_F1': float(per['F1'].mean()), 'weighted_F1': float((per['F1'] * sup).sum() / sup.sum()),
            'cohen_kappa': float(cohen_kappa_score(yt, yp))}
    cml = pd.DataFrame(cm, index=[f'true_{p}' for p in PHASE_NAMES], columns=[f'pred_{p}' for p in PHASE_NAMES]).reset_index().rename(columns={'index': 'true'})
    cml.insert(0, 'model', m)
    return summ, per, cml

res = [phase_table(m) for m in MODELS]
phase_summary = pd.DataFrame([r[0] for r in res]); phase_perclass = pd.concat([r[1] for r in res], ignore_index=True)
phase_conf = pd.concat([r[2] for r in res], ignore_index=True)
phase_summary.to_csv(RESULT_DIR / 'Table_NB18_storage_phase_classification_summary_all_models.csv', index=False)
phase_perclass.to_csv(RESULT_DIR / 'Table_NB18_storage_phase_per_class_metrics_all_models.csv', index=False)
phase_conf.to_csv(RESULT_DIR / 'Table_NB18_storage_phase_confusion_matrices.csv', index=False)
display(phase_summary.round(3))

# validation against the frozen NB09B table
f_ph = find_one('NB09B_storage_phase_classification_summary.csv', required=False)
if f_ph is not None:
    old = pd.read_csv(f_ph).set_index('model'); new = phase_summary.set_index('model')
    common = [m for m in old.index if m in new.index]
    diff = (new.loc[common, ['accuracy', 'balanced_accuracy', 'macro_F1', 'weighted_F1']] - old.loc[common, ['accuracy', 'balanced_accuracy', 'macro_F1', 'weighted_F1']]).abs()
    kap = (new.loc[common, 'cohen_kappa'] - old.loc[common, 'cohen_kappa']).abs()
    print('Validation vs frozen NB09B (max abs difference):', float(max(diff.max().max(), kap.max())))
    if QUICK_TEST:
        print('  (quick test on synthetic data: comparison not meaningful)')
    else:
        assert max(diff.max().max(), kap.max()) < 1e-6, 'Phase classification does not reproduce NB09B - check thresholds/definitions before using new rows.'
        print('PASS - phase classification reproduces the frozen NB09B values.')

# phase-wise MAE / bias / tolerance for all models (Section 3.7 companion table)
ph_rows = []
for m in MODELS:
    for k, name in enumerate(PHASE_NAMES):
        sel = PHASE_OF_DAY(np.arange(22)) == k
        e = (P[mi[m]] - Y)[:, sel].ravel()
        ph_rows.append({'model': m, 'storage_phase': name, 'MAE_days': np.abs(e).mean(), 'bias_days': e.mean(),
                        'within_3d_pct': 100 * np.mean(np.abs(e) <= 3), 'n': e.size})
pd.DataFrame(ph_rows).to_csv(RESULT_DIR / 'Table_NB18_phase_MAE_bias_tolerance_all_models.csv', index=False)

,model,accuracy,balanced_accuracy,macro_F1,weighted_F1,cohen_kappa
0,SVR,0.786,0.786,0.788,0.792,0.680
1,PLSR,0.768,0.766,0.769,0.774,0.653
2,ANN,0.748,0.744,0.748,0.752,0.622
3,CNN1D,0.467,0.466,0.464,0.469,0.205
4,CNN1D_FLAT,0.742,0.737,0.740,0.746,0.613
5,CNN1D_GAP_POS,0.429,0.432,0.419,0.422,0.151
6,BiLSTM,0.468,0.467,0.450,0.455,0.205
7,LSTM,0.448,0.453,0.397,0.401,0.182
8,SimpleRNN,0.376,0.382,0.279,0.282,0.076


Validation vs frozen NB09B (max abs difference): 2.220446049250313e-16
PASS - phase classification reproduces the frozen NB09B values.


## 4. SVR (C, γ) surface from the frozen inner-search logs
The primary NB03 grid (C up to 10⁵) and the wider NB09B grid (C up to 3·10⁶, γ down to 10⁻⁶) are summarised as *mean inner-CV MAE*
(averaged over the five outer folds, minimum over ε for each (C, γ)). A flat region around the selected cell shows that the boundary selections
are not sitting on a steep slope.

In [8]:
f03 = find_one('NB03_inner_search_summary.csv', required=False)
f09 = find_one('NB09B_SVR_sensitivity_inner_search.csv', required=False)
surfaces = {}
if f03 is not None:
    s = pd.read_csv(f03); s = s[(s['model'] == 'SVR') & (s['preprocessing'] == 'sg_deriv1')]
    g = s.groupby(['outer_fold', 'C', 'gamma'])['mean_inner_MAE_days'].min().reset_index()
    surfaces['NB03 primary grid'] = g.groupby(['C', 'gamma'])['mean_inner_MAE_days'].mean().unstack('gamma')
if f09 is not None:
    w = pd.read_csv(f09)
    g = w.groupby(['outer_fold', 'C', 'epsilon', 'gamma'])['MAE_days'].mean().reset_index()
    g = g.groupby(['outer_fold', 'C', 'gamma'])['MAE_days'].min().reset_index()
    surfaces['NB09B wider grid'] = g.groupby(['C', 'gamma'])['MAE_days'].mean().unstack('gamma')

def fmt_sci(v):
    v = float(v); e = int(np.floor(np.log10(v))); m = v / 10 ** e
    return f'$10^{{{e}}}$' if abs(m - 1) < 1e-9 else f'${m:.0f}\\times10^{{{e}}}$'

if surfaces:
    LABEL = {'NB03 primary grid': 'Primary grid', 'NB09B wider grid': 'Wider post hoc grid'}
    allv = np.concatenate([M.values.ravel() for M in surfaces.values()]); vmin = float(np.nanmin(allv)); VMAX = 3.0
    fig, axes = plt.subplots(1, len(surfaces), figsize=(6.5, 3.0), constrained_layout=True)
    axes = np.atleast_1d(axes); plateau_rows = []
    for ax, (name, M) in zip(axes, surfaces.items()):
        M = M.sort_index(ascending=True)
        im = ax.imshow(M.values, origin='lower', aspect='auto', cmap='viridis_r', vmin=vmin, vmax=VMAX)
        ax.set_xticks(range(M.shape[1])); ax.set_xticklabels([fmt_sci(c) for c in M.columns], fontsize=6.5)
        ax.set_yticks(range(M.shape[0])); ax.set_yticklabels([fmt_sci(c) for c in M.index], fontsize=6.5)
        best = np.unravel_index(np.nanargmin(M.values), M.shape); ax.plot(best[1], best[0], marker='*', color='white', ms=10, mec='k')
        mn = np.nanmin(M.values)
        ax.text(best[1], best[0] - 0.42, f'{mn:.3f}', ha='center', va='top', fontsize=6, color='white' if mn > vmin + 0.05 else 'black')
        ax.set_xlabel(r'$\gamma$'); ax.set_ylabel('C'); ax.set_title(LABEL.get(name, name), fontsize=8.5)
        plateau_rows.append({'grid': name, 'best_C': float(M.index[best[0]]), 'best_gamma': float(M.columns[best[1]]), 'best_mean_inner_MAE': float(mn),
                             'n_cells': int(np.isfinite(M.values).sum()),
                             'n_cells_within_0.05d_of_best': int((M.values <= mn + 0.05).sum()),
                             'n_cells_within_0.10d_of_best': int((M.values <= mn + 0.10).sum())})
        M.to_csv(RESULT_DIR / f'Table_NB18_SVR_surface_{name.split()[0]}.csv')
    cb = fig.colorbar(im, ax=axes, shrink=0.9, extend='max', pad=0.02); cb.set_label('Mean inner-CV MAE (days)', fontsize=8)
    save_fig(fig, 'NB18_Figure_SVR_C_gamma_surface')
    pl = pd.DataFrame(plateau_rows); pl.to_csv(RESULT_DIR / 'Table_NB18_SVR_plateau_summary.csv', index=False); display(pl.round(3))
else:
    print('Inner-search logs not found; SVR surface skipped.')

  note: 3 copies of NB03_inner_search_summary.csv; using 05_RESULTS/NB03_CHEMOMETRIC_BASELINES/NB03_inner_search_summary.csv
  figure written: NB18_Figure_SVR_C_gamma_surface.png


,grid,best_C,best_gamma,best_mean_inner_MAE,n_cells,n_cells_within_0.05d_of_best,n_cells_within_0.10d_of_best
0,NB03 primary grid,100000.0,0.0,2.304,42,3,4
1,NB09B wider grid,1000000.0,0.0,2.274,25,7,13


In [9]:
# ------------------------------ wording anchors, protocol, package ------------------------------
def f(x, n=3): return f'{x:.{n}f}'
cs = calib.set_index('model'); A = []
for m in KEY_MODELS:
    r = cs.loc[m]
    A.append(f'{m}: predicted-on-observed slope {f(r.slope_pred_on_obs)} (95% CI {f(r.slope_pred_on_obs_CI_low)}-{f(r.slope_pred_on_obs_CI_high)}); '
             f'calibration slope (observed on predicted) {f(r.slope_obs_on_pred)} (95% CI {f(r.slope_obs_on_pred_CI_low)}-{f(r.slope_obs_on_pred_CI_high)}); '
             f'pooled R2 {f(r.pooled_R2)}.')
lt = phase_bias_tab[(phase_bias_tab['phase'] == 'Late (15-21)') & phase_bias_tab['model'].isin(KEY_MODELS)]
for _, r in lt.iterrows():
    A.append(f'{r.model}, late phase: observed bias {f(r.observed_bias_days,2)} days; bias implied by the linear slope {f(r.bias_implied_by_linear_slope_days,2)} days '
             f'({100*r.share_explained_by_linear_slope:.0f}% of the observed bias); remainder {f(r.residual_beyond_linear_days,2)} days '
             f'(95% CI {f(r.residual_CI_low,2)} to {f(r.residual_CI_high,2)}).')
for m in CNN_MODELS:
    r = phase_summary.set_index('model').loc[m]
    A.append(f'{m}: three-phase accuracy {f(r.accuracy)}, balanced accuracy {f(r.balanced_accuracy)}, macro-F1 {f(r.macro_F1)}, Cohen kappa {f(r.cohen_kappa)}.')
(RESULT_DIR / 'NB18_wording_anchors.md').write_text('# NB18 wording anchors (generated from saved CSV files)\n\n' + '\n\n'.join(f'{i+1}. {s}' for i, s in enumerate(A)), encoding='utf-8')
print('\n'.join(A))

protocol = {'notebook': 'NB18_CALIBRATION_PHASES_AND_SVR_SURFACE', 'run_revision': RUN_REVISION, 'quick_test': QUICK_TEST,
            'analysis_role': 'post hoc analysis of frozen out-of-fold predictions; no model is refitted',
            'models': MODELS, 'statistical_unit': 'egg', 'n_eggs': int(E), 'bootstrap_replicates': BOOT_REPS, 'bootstrap_seed': BOOT_SEED,
            'bootstrap_scheme': 'resample eggs with replacement; retain all 22 rows within each sampled egg',
            'phase_thresholds_days': list(PHASE_EDGES), 'inputs': {'oof': 'NB12_unified_oof_predictions.csv',
            'nb16_oof': str(f16.relative_to(PROJECT_ROOT)) if f16 is not None else None,
            'svr_primary_grid': str(f03.relative_to(PROJECT_ROOT)) if f03 is not None else None,
            'svr_wider_grid': str(f09.relative_to(PROJECT_ROOT)) if f09 is not None else None}, 'predictive_results_modified': False}
(RESULT_DIR / 'NB18_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
(RESULT_DIR / 'NB18_run_summary.json').write_text(json.dumps({'status': 'COMPLETED', 'run_revision': RUN_REVISION,
        'completed_at_utc': datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')
with open(RESULT_DIR / 'environment_packages.txt', 'w', encoding='utf-8') as fh:
    fh.write(f'Python: {sys.version}\nPlatform: {platform.platform()}\n\n')
    try: fh.write(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, stderr=subprocess.STDOUT))
    except Exception as e: fh.write(f'pip freeze failed: {e}\n')

ZIP_NAME = 'NB18_RESULTS_CALIBRATION_PHASES_AND_SVR_SURFACE' + ('_QUICKTEST' if QUICK_TEST else '')
zip_base = ZIP_DIR / ZIP_NAME
if zip_base.with_suffix('.zip').exists(): zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', root_dir=RESULT_DIR)
print('ZIP created:', zip_base.with_suffix('.zip'), '|', round(zip_base.with_suffix('.zip').stat().st_size / 1024, 1), 'KB')
print('NB18 SUCCESS')
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_base.with_suffix('.zip')))

SVR: predicted-on-observed slope 0.855 (95% CI 0.824-0.885); calibration slope (observed on predicted) 0.958 (95% CI 0.924-0.995); pooled R2 0.817.
PLSR: predicted-on-observed slope 0.845 (95% CI 0.820-0.872); calibration slope (observed on predicted) 0.945 (95% CI 0.899-0.994); pooled R2 0.796.
ANN: predicted-on-observed slope 0.823 (95% CI 0.787-0.860); calibration slope (observed on predicted) 0.952 (95% CI 0.895-1.009); pooled R2 0.780.
SVR, late phase: observed bias -1.19 days; bias implied by the linear slope -0.94 days (79% of the observed bias); remainder -0.25 days (95% CI -0.36 to -0.14).
PLSR, late phase: observed bias -1.48 days; bias implied by the linear slope -1.15 days (78% of the observed bias); remainder -0.33 days (95% CI -0.47 to -0.19).
ANN, late phase: observed bias -1.75 days; bias implied by the linear slope -1.49 days (85% of the observed bias); remainder -0.27 days (95% CI -0.43 to -0.10).
CNN1D: three-phase accuracy 0.467, balanced accuracy 0.466, macro-F1 0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>